In [1]:
import os

import pickle

import random

import pandas as pd

from utils.feature_utils import df_to_sequence_array

from config.feature_config import FeatureConfig

### --- Load Dataset ---

In [2]:
random.seed(42)

In [3]:
df = pd.read_excel(
    "../../../data/bpic12.xlsx",
    engine="openpyxl",
    keep_default_na=False,
    dtype={
        "case:concept:name": "string",
        "concept:name": "string",
        "lifecycle:transition": "string",
        "org:resource": "string",
        "case:REG_DATE_HR": "string",
        "case:REG_DATE_DAY": "string",
        "case:REG_DATE_MON": "string",
        "case:AMOUNT_REQ": "float32",
        "time_delta": "float32",
    }
)

df["time:timestamp"] = pd.to_datetime(df["time:timestamp"])

In [4]:
df.head(20)

,case:concept:name,time:timestamp,case:AMOUNT_REQ,case:REG_DATE_DAY,case:REG_DATE_HR,case:REG_DATE_MON,concept:name,lifecycle:transition,org:resource,time_delta
0,173688,2011-10-01 00:38:44.546,20000.0,Saturday,12 AM,October,A_SUBMITTED,COMPLETE,112,0.000000
1,173688,2011-10-01 00:38:44.880,20000.0,Saturday,12 AM,October,A_PARTLYSUBMITTED,COMPLETE,112,0.334000
2,173688,2011-10-01 00:39:37.906,20000.0,Saturday,12 AM,October,A_PREACCEPTED,COMPLETE,112,53.026001
3,173688,2011-10-01 11:42:43.308,20000.0,Saturday,12 AM,October,A_ACCEPTED,COMPLETE,10862,39785.402344
4,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,O_SELECTED,COMPLETE,10862,145.934998
5,173688,2011-10-01 11:45:09.243,20000.0,Saturday,12 AM,October,A_FINALIZED,COMPLETE,10862,0.000000
6,173688,2011-10-01 11:45:11.197,20000.0,Saturday,12 AM,October,O_CREATED,COMPLETE,10862,1.954000
7,173688,2011-10-01 11:45:11.380,20000.0,Saturday,12 AM,October,O_SENT,COMPLETE,10862,0.183000
8,173688,2011-10-10 11:33:03.668,20000.0,Saturday,12 AM,October,O_SENT_BACK,COMPLETE,11049,776872.312500
9,173688,2011-10-13 10:37:29.226,20000.0,Saturday,12 AM,October,A_REGISTERED,COMPLETE,10629,255865.562500


In [6]:
feature_config = FeatureConfig.load(
    path = "../pretrained_models/"
)

### --- Analysis ---

In [7]:
# --- Trace dictionary ---
case_to_trace = (
    df.groupby("case:concept:name")["concept:name"]
      .apply(list)
      .to_dict()
)

# --- Consecutive pattern matcher ---
def has_consecutive_subsequence(trace, pattern):
    L = len(pattern)
    if len(trace) < L:
        return False
    return trace[:L] == pattern

#### 1. A_ACCEPTED Milestone

In [8]:
# --- Undesired patterns ---
pattern1_without_a_preaccepted = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_DECLINED"
]

pattern2_without_a_preaccepted = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_CANCELLED"
]

pattern1_without_a_accepted = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_DECLINED"
]

pattern2_without_a_accepted = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_CANCELLED"
]

# --- Conforming patterns ---
pattern_with_a_preaccepted = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED"
]

pattern_with_a_accepted = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED"
]

In [9]:
cases_with_A_ACCEPTED = set(
    df.loc[df["concept:name"] == "A_ACCEPTED", "case:concept:name"]
)

candidate_cases = [
    cid for cid in case_to_trace.keys()
    if cid not in cases_with_A_ACCEPTED
]

conformant_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_with_A_ACCEPTED
]

In [10]:
groups_without_a_accepted = {k: [] for k in [
    "1_without_a_accepted", "2_without_a_accepted", "1_without_a_preaccepted", "2_without_a_preaccepted", "invalid"
]}

for cid in candidate_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_without_a_accepted):
        groups_without_a_accepted["1_without_a_accepted"].append(cid)

    elif has_consecutive_subsequence(trace, pattern2_without_a_accepted):
        groups_without_a_accepted["2_without_a_accepted"].append(cid)

    elif has_consecutive_subsequence(trace, pattern1_without_a_preaccepted):
        groups_without_a_accepted["1_without_a_preaccepted"].append(cid)

    elif has_consecutive_subsequence(trace, pattern2_without_a_preaccepted):
        groups_without_a_accepted["2_without_a_preaccepted"].append(cid)

    else:
        groups_without_a_accepted["invalid"].append(cid)
   

# --- Sample 10 per group ---
selected_cases_without_a_accepted = {
    k: random.sample(v, min(10, len(v)))
    for k, v in groups_without_a_accepted.items()
}

print("Counts per group:")
for k, v in groups_without_a_accepted.items():
    print(k, len(v))

print("\nSelected cases:")
for k, v in selected_cases_without_a_accepted.items():
    print(k, v)

Counts per group:
1_without_a_accepted 1085
2_without_a_accepted 1100
1_without_a_preaccepted 5719
2_without_a_preaccepted 1
invalid 69

Selected cases:
1_without_a_accepted ['182077', '175714', '194683', '192362', '190770', '184228', '181369', '180400', '205652', '176066']
2_without_a_accepted ['175675', '180968', '190606', '192046', '210881', '175537', '189430', '204460', '190911', '206648']
1_without_a_preaccepted ['208142', '189337', '174114', '183796', '198038', '192840', '189322', '183540', '186397', '192647']
2_without_a_preaccepted ['193378']
invalid ['210104', '212908', '210128', '212605', '212569', '211841', '213064', '208796', '212806', '211552']


In [11]:
sample_case_df = df[df['case:concept:name'] == '213534']
print(sample_case_df)

      case:concept:name          time:timestamp  case:AMOUNT_REQ  \
90369            213534 2012-02-27 21:38:05.582          14500.0   
90370            213534 2012-02-27 21:38:05.742          14500.0   
90371            213534 2012-02-28 09:19:01.301          14500.0   

      case:REG_DATE_DAY case:REG_DATE_HR case:REG_DATE_MON       concept:name  \
90369            Monday            09 PM          February        A_SUBMITTED   
90370            Monday            09 PM          February  A_PARTLYSUBMITTED   
90371            Monday            09 PM          February      A_PREACCEPTED   

      lifecycle:transition org:resource    time_delta  
90369             COMPLETE          112      0.000000  
90370             COMPLETE          112      0.160000  
90371             COMPLETE        10881  42055.558594  


In [12]:
groups_with_a_accepted = {k: [] for k in [
    "with_a_preaccepted", "with_a_accepted"
]}

for cid in conformant_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern_with_a_accepted):
        groups_with_a_accepted["with_a_accepted"].append(cid)

    if has_consecutive_subsequence(trace, pattern_with_a_preaccepted):
        groups_with_a_accepted["with_a_preaccepted"].append(cid)

print("Counts per group:")
for k, v in groups_with_a_accepted.items():
    print(k, len(v))

# print("\nSelected cases:")
# for k, v in groups_with_a_accepted.items():
#     print(k, v)

Counts per group:
with_a_preaccepted 5113
with_a_accepted 5113


In [13]:
sample_case_df = df[df['case:concept:name'] == '173688']
print(sample_case_df)

   case:concept:name          time:timestamp  case:AMOUNT_REQ  \
0             173688 2011-10-01 00:38:44.546          20000.0   
1             173688 2011-10-01 00:38:44.880          20000.0   
2             173688 2011-10-01 00:39:37.906          20000.0   
3             173688 2011-10-01 11:42:43.308          20000.0   
4             173688 2011-10-01 11:45:09.243          20000.0   
5             173688 2011-10-01 11:45:09.243          20000.0   
6             173688 2011-10-01 11:45:11.197          20000.0   
7             173688 2011-10-01 11:45:11.380          20000.0   
8             173688 2011-10-10 11:33:03.668          20000.0   
9             173688 2011-10-13 10:37:29.226          20000.0   
10            173688 2011-10-13 10:37:29.226          20000.0   
11            173688 2011-10-13 10:37:29.226          20000.0   
12            173688 2011-10-13 10:37:29.226          20000.0   

   case:REG_DATE_DAY case:REG_DATE_HR case:REG_DATE_MON       concept:name  \
0          

In [14]:
candidate_arr_1_without_a_preaccepted = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_accepted["1_without_a_preaccepted"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_2_without_a_preaccepted = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_accepted["2_without_a_preaccepted"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

conformant_arr_with_a_preaccepted = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(groups_with_a_accepted["with_a_preaccepted"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp",
    prefix_len=3
)

In [15]:
candidate_arr_1_without_a_accepted = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_accepted["1_without_a_accepted"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_2_without_a_accepted = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_accepted["2_without_a_accepted"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

conformant_arr_with_a_accepted = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(groups_with_a_accepted["with_a_accepted"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp",
    prefix_len=4
)

#### 2. A_FINALIZED Milestone

In [16]:
# --- Undesired patterns ---
pattern1_without_a_finalized = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_DECLINED"
]

pattern2_without_a_finalized = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_CANCELLED"
]

# --- Conforming patterns ---
pattern1_with_a_finalized = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED"
]

pattern2_with_a_finalized = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED"
]

In [17]:
cases_with_A_FINALIZED = set(
    df.loc[df["concept:name"] == "A_FINALIZED", "case:concept:name"]
)
cases_without_A_FINALIZED = list(cases_with_A_ACCEPTED - cases_with_A_FINALIZED)

candidate_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_without_A_FINALIZED
]

conformant_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_with_A_FINALIZED
]

In [18]:
groups_without_a_finalized = {k: [] for k in [
    "1_without_a_finalized", "2_without_a_finalized", "invalid"
]}

for cid in candidate_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_without_a_finalized):
        groups_without_a_finalized["1_without_a_finalized"].append(cid)

    elif has_consecutive_subsequence(trace, pattern2_without_a_finalized):
        groups_without_a_finalized["2_without_a_finalized"].append(cid)

    else:
        groups_without_a_finalized["invalid"].append(cid)
   

# --- Sample 10 per group ---
selected_cases_without_a_finalized = {
    k: random.sample(v, min(10, len(v)))
    for k, v in groups_without_a_finalized.items()
}

print("Counts per group:")
for k, v in groups_without_a_finalized.items():
    print(k, len(v))

print("\nSelected cases:")
for k, v in selected_cases_without_a_finalized.items():
    print(k, v)

Counts per group:
1_without_a_finalized 29
2_without_a_finalized 66
invalid 3

Selected cases:
1_without_a_finalized ['199351', '178152', '194182', '177185', '211931', '189775', '201517', '200611', '194013', '200221']
2_without_a_finalized ['188720', '177137', '176398', '202346', '180580', '205559', '182578', '212058', '207978', '211483']
invalid ['210452', '211197', '213267']


In [19]:
sample_case_df = df[df['case:concept:name'] == "195923"]
print(sample_case_df)

      case:concept:name          time:timestamp  case:AMOUNT_REQ  \
49901            195923 2011-12-27 20:31:16.638           3987.0   
49902            195923 2011-12-27 20:31:16.786           3987.0   
49903            195923 2011-12-27 20:31:52.925           3987.0   
49904            195923 2011-12-29 09:30:12.299           3987.0   
49905            195923 2011-12-29 09:30:33.042           3987.0   

      case:REG_DATE_DAY case:REG_DATE_HR case:REG_DATE_MON       concept:name  \
49901           Tuesday            08 PM          December        A_SUBMITTED   
49902           Tuesday            08 PM          December  A_PARTLYSUBMITTED   
49903           Tuesday            08 PM          December      A_PREACCEPTED   
49904           Tuesday            08 PM          December         A_ACCEPTED   
49905           Tuesday            08 PM          December         A_DECLINED   

      lifecycle:transition org:resource  time_delta  
49901             COMPLETE          112       0.00

In [20]:
groups_with_a_finalized = {k: [] for k in [
    "with_a_finalized"
]}

for cid in conformant_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_with_a_finalized) or has_consecutive_subsequence(trace, pattern2_with_a_finalized):
        groups_with_a_finalized["with_a_finalized"].append(cid)

print("Counts per group:")
for k, v in groups_with_a_finalized.items():
    print(k, len(v))

# print("\nSelected cases:")
# for k, v in groups_with_a_finalized.items():
#     print(k, v)

Counts per group:
with_a_finalized 5015


In [21]:
sample_case_df = df[df['case:concept:name'] == "173688"]
print(sample_case_df)

   case:concept:name          time:timestamp  case:AMOUNT_REQ  \
0             173688 2011-10-01 00:38:44.546          20000.0   
1             173688 2011-10-01 00:38:44.880          20000.0   
2             173688 2011-10-01 00:39:37.906          20000.0   
3             173688 2011-10-01 11:42:43.308          20000.0   
4             173688 2011-10-01 11:45:09.243          20000.0   
5             173688 2011-10-01 11:45:09.243          20000.0   
6             173688 2011-10-01 11:45:11.197          20000.0   
7             173688 2011-10-01 11:45:11.380          20000.0   
8             173688 2011-10-10 11:33:03.668          20000.0   
9             173688 2011-10-13 10:37:29.226          20000.0   
10            173688 2011-10-13 10:37:29.226          20000.0   
11            173688 2011-10-13 10:37:29.226          20000.0   
12            173688 2011-10-13 10:37:29.226          20000.0   

   case:REG_DATE_DAY case:REG_DATE_HR case:REG_DATE_MON       concept:name  \
0          

In [22]:
candidate_arr_1_without_a_finalized = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_finalized["1_without_a_finalized"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_2_without_a_finalized = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_finalized["2_without_a_finalized"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

conformant_arr_with_a_finalized = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(groups_with_a_finalized["with_a_finalized"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp",
    prefix_len=5,
)

#### 3. A_APPROVED Milestone

In [23]:
cases_with_A_APPROVED = set(
    df.loc[df["concept:name"] == "A_APPROVED", "case:concept:name"]
)
cases_without_A_APPROVED = list(cases_with_A_FINALIZED - cases_with_A_APPROVED)

candidate_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_without_A_APPROVED
]

conformant_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_with_A_APPROVED
]

In [24]:
from collections import Counter

sequences = []

for case_id in cases_without_A_APPROVED:
    events = df.loc[
        df["case:concept:name"] == case_id,
        "concept:name"
    ].tolist()
    
    sequences.append(tuple(events))

unique_sequences = Counter(sequences)

for seq, count in unique_sequences.most_common():
    print(f"\nCount: {count}")
    print(" → ".join(seq))


Count: 334
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → A_CANCELLED → O_CANCELLED

Count: 320
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → O_CANCELLED → A_CANCELLED

Count: 255
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → A_FINALIZED → O_SELECTED → O_CREATED → O_SENT → A_CANCELLED → O_CANCELLED

Count: 223
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → A_FINALIZED → O_SELECTED → O_CREATED → O_SENT → O_CANCELLED → A_CANCELLED

Count: 191
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → O_SENT_BACK → O_DECLINED → A_DECLINED

Count: 147
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → O_SELECTED → A_FINALIZED → O_CREATED → O_SENT → O_SENT_BACK → A_DECLINED → O_DECLINED

Count: 118
A_SUBMITTED → A_PARTLYSUBMITTED → A_PREACCEPTED → A_ACCEPTED → A_FINAL

In [25]:
# --- Undesired patterns ---
pattern1_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "A_CANCELLED",
    "O_CANCELLED"
]

pattern2_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "A_CANCELLED",
    "O_CANCELLED"
]

pattern3_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_CANCELLED",
    "A_CANCELLED",
]

pattern4_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_CANCELLED",
    "A_CANCELLED",
]

# --- Conforming patterns ---
pattern1_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "A_APPROVED"
]

pattern2_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "A_APPROVED"
]

pattern3_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED"
]

pattern4_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED"
]

In [26]:
# --- Undesired patterns ---
pattern5_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "A_DECLINED",
    "O_DECLINED"
]

pattern6_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "A_DECLINED",
    "O_DECLINED"
]

pattern7_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_DECLINED",
    "A_DECLINED",
]

pattern8_without_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_DECLINED",
    "A_DECLINED",
]

# --- Conforming patterns ---
pattern5_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_APPROVED"
]

pattern6_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED"
    "A_APPROVED"  
]

pattern7_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "O_SELECTED",
    "A_FINALIZED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "A_APPROVED",
    "O_ACCEPTED"
]

pattern8_with_a_approved = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "A_APPROVED",
    "O_ACCEPTED"
]

In [27]:
groups_without_a_approved = {k: [] for k in [
    "1_without_a_approved", "2_without_a_approved", "3_without_a_approved", "4_without_a_approved", 
    "5_without_a_approved", "6_without_a_approved", "7_without_a_approved", "8_without_a_approved", 
    "invalid"
]}

for cid in candidate_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_without_a_approved):
        groups_without_a_approved["1_without_a_approved"].append(cid)

    elif has_consecutive_subsequence(trace, pattern2_without_a_approved):
        groups_without_a_approved["2_without_a_approved"].append(cid)

    if has_consecutive_subsequence(trace, pattern3_without_a_approved):
        groups_without_a_approved["3_without_a_approved"].append(cid)

    elif has_consecutive_subsequence(trace, pattern4_without_a_approved):
        groups_without_a_approved["4_without_a_approved"].append(cid)
        
    elif has_consecutive_subsequence(trace, pattern5_without_a_approved):
        groups_without_a_approved["5_without_a_approved"].append(cid)
        
    elif has_consecutive_subsequence(trace, pattern6_without_a_approved):
        groups_without_a_approved["6_without_a_approved"].append(cid)
        
    elif has_consecutive_subsequence(trace, pattern7_without_a_approved):
        groups_without_a_approved["7_without_a_approved"].append(cid)
        
    elif has_consecutive_subsequence(trace, pattern8_without_a_approved):
        groups_without_a_approved["8_without_a_approved"].append(cid)

    else:
        groups_without_a_approved["invalid"].append(cid)
   

# --- Sample 10 per group ---
selected_cases_without_a_approved = {
    k: random.sample(v, min(10, len(v)))
    for k, v in groups_without_a_approved.items()
}

print("Counts per group:")
for k, v in groups_without_a_approved.items():
    print(k, len(v))

print("\nSelected cases:")
for k, v in selected_cases_without_a_approved.items():
    print(k, v)

Counts per group:
1_without_a_approved 334
2_without_a_approved 255
3_without_a_approved 320
4_without_a_approved 223
5_without_a_approved 147
6_without_a_approved 118
7_without_a_approved 191
8_without_a_approved 118
invalid 1652

Selected cases:
1_without_a_approved ['199041', '212346', '194692', '182500', '194973', '194278', '184889', '189481', '214034', '176825']
2_without_a_approved ['196951', '197981', '180721', '195178', '201821', '182864', '180046', '192761', '189763', '184610']
3_without_a_approved ['206669', '185003', '192752', '176738', '185563', '175720', '192088', '197464', '188783', '177092']
4_without_a_approved ['181658', '197969', '204119', '186751', '202122', '194716', '190983', '200973', '192611', '179110']
5_without_a_approved ['189328', '180754', '187214', '212298', '209511', '201602', '199411', '196960', '185078', '208136']
6_without_a_approved ['195881', '178789', '206790', '175994', '210617', '179284', '180538', '199717', '180817', '208403']
7_without_a_approved

In [28]:
sample_case_df = df[df['case:concept:name'] == "203702"]
print(sample_case_df)

      case:concept:name          time:timestamp  case:AMOUNT_REQ  \
68318            203702 2012-01-24 19:16:21.898          10500.0   
68319            203702 2012-01-24 19:16:22.106          10500.0   
68320            203702 2012-01-24 19:17:00.272          10500.0   
68321            203702 2012-01-24 19:50:23.216          10500.0   
68322            203702 2012-01-24 19:54:34.766          10500.0   
68323            203702 2012-01-24 19:54:34.766          10500.0   
68324            203702 2012-01-24 19:54:35.914          10500.0   
68325            203702 2012-01-24 19:54:35.941          10500.0   
68326            203702 2012-01-31 12:33:16.343          10500.0   
68327            203702 2012-02-06 14:54:05.529          10500.0   
68328            203702 2012-02-06 14:54:05.529          10500.0   
68329            203702 2012-02-06 14:54:05.529          10500.0   
68330            203702 2012-02-06 14:54:05.529          10500.0   

      case:REG_DATE_DAY case:REG_DATE_HR case:R

In [29]:
groups_with_a_approved = {k: [] for k in [
    "1_with_a_approved", "2_with_a_approved"
]}

for cid in conformant_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_with_a_approved) or has_consecutive_subsequence(trace, pattern2_with_a_approved) \
    or has_consecutive_subsequence(trace, pattern3_with_a_approved) or has_consecutive_subsequence(trace, pattern4_with_a_approved):
        groups_with_a_approved["1_with_a_approved"].append(cid)

    if has_consecutive_subsequence(trace, pattern5_with_a_approved) or has_consecutive_subsequence(trace, pattern6_with_a_approved) \
    or has_consecutive_subsequence(trace, pattern7_with_a_approved) or has_consecutive_subsequence(trace, pattern8_with_a_approved):
        groups_with_a_approved["2_with_a_approved"].append(cid)

print("Counts per group:")
for k, v in groups_with_a_approved.items():
    print(k, len(v))

# print("\nSelected cases:")
# for k, v in groups_with_a_approved.items():
#     print(k, v)

Counts per group:
1_with_a_approved 1212
2_with_a_approved 466


In [30]:
# sample_case_df = df[df['case:concept:name'] == "173760"]
print(sample_case_df)

      case:concept:name          time:timestamp  case:AMOUNT_REQ  \
68318            203702 2012-01-24 19:16:21.898          10500.0   
68319            203702 2012-01-24 19:16:22.106          10500.0   
68320            203702 2012-01-24 19:17:00.272          10500.0   
68321            203702 2012-01-24 19:50:23.216          10500.0   
68322            203702 2012-01-24 19:54:34.766          10500.0   
68323            203702 2012-01-24 19:54:34.766          10500.0   
68324            203702 2012-01-24 19:54:35.914          10500.0   
68325            203702 2012-01-24 19:54:35.941          10500.0   
68326            203702 2012-01-31 12:33:16.343          10500.0   
68327            203702 2012-02-06 14:54:05.529          10500.0   
68328            203702 2012-02-06 14:54:05.529          10500.0   
68329            203702 2012-02-06 14:54:05.529          10500.0   
68330            203702 2012-02-06 14:54:05.529          10500.0   

      case:REG_DATE_DAY case:REG_DATE_HR case:R

In [31]:
candidate_arr_1_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["1_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_2_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["2_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_3_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["3_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_4_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["4_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_5_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["5_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_6_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["6_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_7_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["7_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_8_without_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_without_a_approved["8_without_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

conformant_arr_1_with_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(groups_with_a_approved["1_with_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp",
    prefix_len=10
)

conformant_arr_2_with_a_approved = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(groups_with_a_approved["2_with_a_approved"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp",
    prefix_len=11
)

#### 4. Swapped Ordering of A_APPROVED A_REGISTERED

In [32]:
candidate_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_with_A_APPROVED
]

conformant_cases = [
    cid for cid in case_to_trace.keys()
    if cid in cases_with_A_APPROVED
]

In [33]:
# --- Undesired patterns ---
pattern1_swap = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_REGISTERED",
    "A_APPROVED",
    "A_ACTIVATED"
]

pattern2_swap = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_ACTIVATED",
    "A_APPROVED",
    "A_REGISTERED"
]

pattern3_swap = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_REGISTERED",
    "A_ACTIVATED",
    "A_APPROVED"
]

pattern4_swap = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_ACTIVATED",
    "A_REGISTERED",
    "A_APPROVED"
]

# --- Conforming patterns ---
pattern1_nonswap = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_APPROVED",
    "A_REGISTERED",
    "A_ACTIVATED"
]

pattern2_nonswap = [
    "A_SUBMITTED",
    "A_PARTLYSUBMITTED",
    "A_PREACCEPTED",
    "A_ACCEPTED",
    "A_FINALIZED",
    "O_SELECTED",
    "O_CREATED",
    "O_SENT",
    "O_SENT_BACK",
    "O_ACCEPTED",
    "A_APPROVED",
    "A_ACTIVATED",
    "A_REGISTERED"
]

In [34]:
groups_swap = {k: [] for k in [
    "1_swap", "2_swap", "3_swap", "4_swap"
]}

for cid in candidate_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_swap):
        groups_swap["1_swap"].append(cid)

    elif has_consecutive_subsequence(trace, pattern2_swap):
        groups_swap["2_swap"].append(cid)

    if has_consecutive_subsequence(trace, pattern3_swap):
        groups_swap["3_swap"].append(cid)

    elif has_consecutive_subsequence(trace, pattern4_swap):
        groups_swap["4_swap"].append(cid)
   

# --- Sample 10 per group ---
selected_cases_swap = {
    k: random.sample(v, min(10, len(v)))
    for k, v in groups_swap.items()
}

print("Counts per group:")
for k, v in groups_swap.items():
    print(k, len(v))

print("\nSelected cases:")
for k, v in selected_cases_swap.items():
    print(k, v)

Counts per group:
1_swap 93
2_swap 52
3_swap 36
4_swap 21

Selected cases:
1_swap ['186872', '202089', '182960', '180445', '208139', '189244', '209302', '206546', '183703', '182074']
2_swap ['187942', '210587', '178739', '194031', '193174', '173913', '199519', '183238', '192383', '174255']
3_swap ['180025', '198170', '194508', '190216', '178783', '213010', '204203', '194499', '176137', '203122']
4_swap ['203495', '180313', '206865', '182668', '212061', '213300', '195761', '212274', '206408', '201833']


In [35]:
sample_case_df = df[df['case:concept:name'] == "182074"]
print(sample_case_df)

      case:concept:name          time:timestamp  case:AMOUNT_REQ  \
19777            182074 2011-11-03 20:46:41.218          20000.0   
19778            182074 2011-11-03 20:46:41.419          20000.0   
19779            182074 2011-11-03 20:47:21.541          20000.0   
19780            182074 2011-11-03 20:49:17.401          20000.0   
19781            182074 2011-11-03 20:50:34.322          20000.0   
19782            182074 2011-11-03 20:50:34.322          20000.0   
19783            182074 2011-11-03 20:50:35.172          20000.0   
19784            182074 2011-11-03 20:50:35.199          20000.0   
19785            182074 2011-11-03 20:52:02.250          20000.0   
19786            182074 2011-11-07 13:25:11.610          20000.0   
19787            182074 2011-11-07 13:25:11.610          20000.0   
19788            182074 2011-11-07 13:25:11.610          20000.0   
19789            182074 2011-11-07 13:25:11.610          20000.0   

      case:REG_DATE_DAY case:REG_DATE_HR case:R

In [36]:
groups_nonswap = {k: [] for k in [
    "nonswap"
]}

for cid in conformant_cases:
    trace = case_to_trace[cid]

    if has_consecutive_subsequence(trace, pattern1_nonswap) or has_consecutive_subsequence(trace, pattern2_nonswap):
        groups_nonswap["nonswap"].append(cid)

print("Counts per group:")
for k, v in groups_nonswap.items():
    print(k, len(v))

# print("\nSelected cases:")
# for k, v in groups_nonswap.items():
#     print(k, v)

Counts per group:
nonswap 150


In [37]:
candidate_arr_1_swap = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_swap["1_swap"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_2_swap = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_swap["2_swap"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_3_swap = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_swap["3_swap"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

candidate_arr_4_swap = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(selected_cases_swap["4_swap"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp"
)

conformant_arr_nonswap = df_to_sequence_array(
    df=df[df["case:concept:name"].isin(groups_nonswap["nonswap"])],
    case_id_field="case:concept:name",
    feature_config=feature_config,
    sort_field="time:timestamp",
    prefix_len=13
)

### --- Save ---

In [38]:
experiments = {
    "a_preaccepted": {
        "candidates": [candidate_arr_1_without_a_preaccepted, candidate_arr_2_without_a_preaccepted],
        "conformant": conformant_arr_with_a_preaccepted,
    },
    
    "a_accepted": {
        "candidates": [candidate_arr_1_without_a_accepted, candidate_arr_2_without_a_accepted],
        "conformant": conformant_arr_with_a_accepted,
    },
    
    "a_finalized": {
        "candidates": [candidate_arr_1_without_a_finalized, candidate_arr_2_without_a_finalized],
        "conformant": conformant_arr_with_a_finalized,
    },

    "1_a_approved": {
        "candidates": [candidate_arr_1_without_a_approved, candidate_arr_2_without_a_approved, 
                       candidate_arr_3_without_a_approved, candidate_arr_4_without_a_approved],
        "conformant": conformant_arr_1_with_a_approved,
    },
   
    "2_a_approved": {
        "candidates": [candidate_arr_5_without_a_approved, candidate_arr_6_without_a_approved,
                      candidate_arr_7_without_a_approved, candidate_arr_8_without_a_approved],
        "conformant": conformant_arr_2_with_a_approved,
    },

    "1_swap": {
        "candidates": [candidate_arr_1_swap],
        "conformant": conformant_arr_nonswap,
    },
    "2_swap": {
        "candidates": [candidate_arr_2_swap],
        "conformant": conformant_arr_nonswap,
    },
    "3_swap": {
        "candidates": [candidate_arr_3_swap],
        "conformant": conformant_arr_nonswap,
    },
    "4_swap": {
        "candidates": [candidate_arr_4_swap],
        "conformant": conformant_arr_nonswap,
    },
}

In [39]:
experiments_metadata = {
    "prefix_lengths": {
        "a_preaccepted": 3,
        "a_accepted": 4,
        "a_finalized": 5,
        "1_a_approved": 10,
        "2_a_approved": 11,
        "1_swap": 13,
        "2_swap": 13,
        "3_swap": 13,
        "4_swap": 13
    },
}

In [40]:
os.makedirs("../experiments", exist_ok=True)
with open("../experiments/cf_domain_examples.pkl", "wb") as f:
    pickle.dump({
    "experiments": experiments,
    "metadata": experiments_metadata
}, f)

### --- Load ---

In [41]:
with open("../experiments/cf_domain_examples.pkl", "rb") as f:
    data = pickle.load(f)

experiments = data["experiments"]
metadata = data["metadata"]

In [42]:
metadata

{'prefix_lengths': {'a_preaccepted': 3,
  'a_accepted': 4,
  'a_finalized': 5,
  '1_a_approved': 10,
  '2_a_approved': 11,
  '1_swap': 13,
  '2_swap': 13,
  '3_swap': 13,
  '4_swap': 13}}